<a href="https://colab.research.google.com/github/Satya-Padala/24P31A42V7_DriveReady_OPC_2026/blob/main/Day-2/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install -q openai

In [31]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Sure, here’s a classic‑but‑still‑fresh one for you:

**Why don't scientists trust atoms anymore?**

*Because they make up everything!* 😄


In [32]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an egg‑less cake for you! Please note we need at least 24 hours notice, so today evening isn’t possible. Let us know your preferred flavor and we’ll schedule it.
CompletionUsage(completion_tokens=294, prompt_tokens=183, total_tokens=477, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=241, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.137893844, prompt_time=0.007574561, completion_time=0.613859696, total_time=0.621434257)


In [17]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can you deliver in 1 hour?"}],
)
print(response.choices[0].message.content)

I’m sorry, but we bake each cake to order and need at least 24 hours’ notice, so we can’t deliver within an hour. All of our cakes also have a delicious egg‑less version if you’d like. Let us know what you’d like, and we’ll schedule it for you!


In [19]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can you deliver in 1 hour?"}],
)
print(response.choices[0].message.content)

We’re sorry, but we need at least 24 hours’ notice to bake a cake, so a one‑hour delivery isn’t possible. All of our cakes can also be made egg‑free if you’d like. Let us know what you’d like, and we’ll schedule it for you!


In [44]:
SYSTEM=""" You are the assistant for a Santhi's pickles shop.
Facts:we prepare it after we get an order;veg and non-veg pickles are available;for one order it takes 5-6 days.
Rules:no deliveries on sunday;if you do not know ,say so.
Format:Reply in a polite way;answer should be under 50words."""


In [35]:
def classify(new_message):
    messages = [
        {"role": "system", "content": "Label the customer message with one word: order, question or complaint."},

        {"role": "user", "content": "need 2 chicken pickles for friday"},
        {"role": "assistant", "content": "order"},

        {"role": "user", "content": "do you have veg pickles?"},
        {"role": "assistant", "content": "question"},

        {"role": "user", "content": "the pickle was not good"},
        {"role": "assistant", "content": "complaint"},

        {"role": "user", "content": new_message},
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
    )

    return response.choices[0].message.content.strip()

In [36]:
print(classify("I want 3 mango pickles"))
print(classify("Do you have prawn pickle?"))
print(classify("I received a damaged bottle"))
print(classify("How many days does preparation take?"))
print(classify("I want to order chicken pickle"))

order
question
complaint
question
order


In [40]:
import json

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Extract order details and return only valid JSON with name, item, qty, date and eggless."
        },
        {
            "role": "user",
            "content": "My name is Ravi. I want 3 mango pickles for 28 November."
        }
    ],
)

try:
    order = json.loads(response.choices[0].message.content)
    print(order)
except json.JSONDecodeError:
    print("Invalid JSON response")

{'name': 'Ravi', 'item': 'mango pickles', 'qty': 3, 'date': '28 November', 'eggless': None}


In [45]:
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 2

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:] # system + last few
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", messages=recent, temperature=0.3)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

In [47]:
print(chat("My name is satya."))
print(chat("What products do you sell?"))
print(chat("How many days does preparation take?"))
print(chat("Do you deliver on Sundays?"))
print(chat("What is my name?"))

Hello Satya! How can I assist you with your pickle order today?
We offer a range of fresh pickles, both vegetarian and non‑vegetarian varieties. Each order is freshly prepared after you place it, ensuring the best flavor and quality. Let me know which type you’re interested in, and I’ll give you more details!
It takes about 5‑6 days from the moment we receive your order to have the pickles ready for delivery. If you need any more details, just let me know!
No, we don’t deliver on Sundays. We’ll arrange delivery for the next business day. Let us know if you’d like to schedule it!
I’m sorry, but I don’t have that information.


In [48]:
PICKLE_CATALOGUE = {
    "Chicken pickle": 1000,
    "Prawn pickle": 900,
    "Mango pickle": 800
}


def get_price(item: str, qty: int = 1) -> int:
    """
    Returns the total price of a pickle based on its quantity.
    """
    return PICKLE_CATALOGUE[item] * qty


SYSTEM="""
Role:You are the assistant for a Santhi's pickles shop.
Facts:
1)we prepare it after we get an order
2)veg and non-veg pickles are available
3)for one order it takes 5-6 days.
Rules:
1)no deliveries on sunday
2)if you do not know ,say so.
Format:
1)Reply in a polite way
2)answer should be under 50 words.
"""